In [2]:
print("Spark:", spark.version)
print("SC:", sc)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark: 3.5.5-amzn-1
SC: <SparkContext master=yarn appName=livy-session-0>

In [3]:
BUCKET = "crypto-bigdata-2026"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"s3://{BUCKET}/raw/csv/")

print("Filas raw:", df_raw.count())
df_raw.printSchema()
df_raw.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Filas raw: 8362
root
 |-- SNo: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Symbol: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Marketcap: double (nullable = true)

+---+-------+------+-------------------+------------------+------------------+------------------+------------------+------+---------------+
|SNo|   Name|Symbol|               Date|              High|               Low|              Open|             Close|Volume|      Marketcap|
+---+-------+------+-------------------+------------------+------------------+------------------+------------------+------+---------------+
|  1|Bitcoin|   BTC|2013-04-29 23:59:59|147.48800659179688|             134.0|134.44400024414062| 144.5399932861328|   0.0| 1.6037688645E9|
|  2|Bitcoin|   BTC|2013-04-30 23:59:59|146.92

In [5]:
from pyspark.sql.functions import (
    col, to_date, year, month,
    round as r, trim
)

df_clean = df_raw \
    .dropna(subset=['Close','Open','High','Low','Date','Name']) \
    .filter(col('Close') > 0) \
    .filter(col('Volume') > 0) \
    .withColumn('Date', to_date(col('Date'))) \
    .withColumn('Year', year(col('Date'))) \
    .withColumn('Month', month(col('Date'))) \
    .withColumn('Name', trim(col('Name'))) \
    .withColumn('daily_return',
        r((col('Close')-col('Open'))/col('Open')*100, 4)) \
    .withColumn('volatility',
        r((col('High')-col('Low'))/col('Close')*100, 4)) \
    .filter(col('Year') >= 2017) \
    .dropDuplicates(['Date','Name'])

print("Filas limpias:", df_clean.count())
df_clean.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Filas limpias: 6507
+----+--------+------+----------+------------------+------------------+------------------+------------------+------------+----------------+----+-----+------------+----------+
| SNo|    Name|Symbol|      Date|              High|               Low|              Open|             Close|      Volume|       Marketcap|Year|Month|daily_return|volatility|
+----+--------+------+----------+------------------+------------------+------------------+------------------+------------+----------------+----+-----+------------+----------+
|1344| Bitcoin|   BTC|2017-01-01|1003.0800170898438|  958.698974609375| 963.6580200195312| 998.3250122070312|1.47775008E8|1.60504074605E10|2017|    1|      3.5974|    4.4456|
| 513|Ethereum|   ETH|2017-01-01| 8.471229553222656| 7.982309818267822| 7.982309818267822|  8.17257022857666|   1.47317E7| 7.15049207868E8|2017|    1|      2.3835|    5.9824|
|1345| Bitcoin|   BTC|2017-01-02|1031.3900146484375| 996.7020263671875| 998.6170043945312|           1021

In [6]:
df_clean.write \
    .mode("overwrite") \
    .option("header", "true") \
    .partitionBy("Name") \
    .csv(f"s3://{BUCKET}/trusted/crypto_prices/")

print("Trusted guardado, particionado por Name")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Trusted guardado, particionado por Name

In [7]:
df_meta = spark.read \
    .option("header", "true") \
    .csv(f"s3://{BUCKET}/raw/rds/coins_metadata.csv")

df_meta.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"s3://{BUCKET}/trusted/coins_metadata/")

print("Metadata guardada en trusted")
df_meta.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Metadata guardada en trusted
+---+------+---------+---------------+----------+-----------+-------------+----------------+------------+
| id|symbol|     name|       category|blockchain|launch_year|   max_supply|       consensus|coingecko_id|
+---+------+---------+---------------+----------+-----------+-------------+----------------+------------+
|  1|   BTC|  Bitcoin| Store of Value|   Bitcoin|       2009|   21000000.0|   Proof of Work|     bitcoin|
|  2|   ETH| Ethereum|Smart Contracts|  Ethereum|       2015|         NULL|  Proof of Stake|    ethereum|
|  3|   SOL|   Solana|Smart Contracts|    Solana|       2020|         NULL|Proof of History|      solana|
|  4|   ADA|  Cardano|Smart Contracts|   Cardano|       2017|45000000000.0|  Proof of Stake|     cardano|
|  5|  LINK|Chainlink| Oracle Network|  Ethereum|       2017| 1000000000.0|      PoS/Oracle|   chainlink|
+---+------+---------+---------------+----------+-----------+-------------+----------------+------------+